# Toy Corpus Implementation and Training

This notebook walks through a complete mini language-model workflow from scratch.

The main idea is simple: we take a tiny text corpus, convert words to numbers, train a small model to predict the next word, and then test whether the model learned useful patterns.

Why this notebook matters:
- It makes the full pipeline visible from raw text to saved model files.
- It keeps everything small, so errors are easy to spot and fix.
- It builds intuition you will reuse for tokenizer work and larger transformer training later.

What you will build in order:
1. Set up imports and training configuration.
2. Build a tiny in-memory corpus.
3. Tokenize and encode text.
4. Create next-token training examples.
5. Define a simple baseline model.
6. Train the model.
7. Evaluate predictions on sample prompts.
8. Save and reload artifacts.

How to read this notebook:
- Read each markdown section first.
- Run the code cell below it.
- Compare the output to what the section says should happen.

If you are new to this, think of the notebook as a toy lab where each step is intentionally small so the logic stays clear.

## How To Use This Notebook

Recommended run order:
1. Run all cells from top to bottom once.
2. Confirm loss decreases in the training section.
3. Inspect sample predictions in the evaluation section.
4. Confirm artifacts are saved and reloaded.

Then try small experiments:
- Increase or decrease `SEQ_LEN`.
- Change `EMBED_DIM` and compare loss trends.
- Add new corpus lines and see how predictions change.

If something breaks, check in this order:
- tokenization and vocabulary mapping
- training-example shapes
- learning rate and epoch settings
- save/reload paths

In [12]:
import json
import pickle
import random
from pathlib import Path

import numpy as np


# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Core training settings
SEQ_LEN = 4
BATCH_SIZE = 8
LEARNING_RATE = 0.1
EPOCHS = 250
EMBED_DIM = 16

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Configuration loaded.")
print(f"SEED={SEED}, SEQ_LEN={SEQ_LEN}, BATCH_SIZE={BATCH_SIZE}, "
      f"LR={LEARNING_RATE}, EPOCHS={EPOCHS}, EMBED_DIM={EMBED_DIM}")

Configuration loaded.
SEED=42, SEQ_LEN=4, BATCH_SIZE=8, LR=0.1, EPOCHS=250, EMBED_DIM=16


## 1) Set Up Imports and Configuration

In this section, we load libraries and set training constants.

What each setting controls:
- SEED: makes random operations repeatable, so your results are easier to debug.
- SEQ_LEN: how many previous tokens the model sees before predicting the next one.
- BATCH_SIZE: how many examples are processed before each parameter update.
- LEARNING_RATE: how large each update step is during training.
- EPOCHS: how many passes over the training data.
- EMBED_DIM: size of each word vector inside the model.

Why this matters:
- Good experiments are reproducible.
- Clear hyperparameters make model behavior easier to interpret.
- Creating an artifacts folder up front ensures save and reload works later without path errors.

## 2) Build the Toy Corpus

Now we create a very small corpus directly in memory.

Why a tiny corpus first:
- Fast runs let you iterate quickly.
- You can manually inspect almost every sentence.
- It reduces noise when you are testing pipeline correctness.

What to look for in the output:
- Number of sentences should match what you entered.
- First few lines should look clean and consistent.
- The text should include repeated words and simple patterns, because repeated structure helps the model learn something measurable.

In [2]:
corpus = [
    "the model learns from text",
    "the tokenizer splits text into pieces",
    "byte pair encoding merges common pairs",
    "small corpora are good for debugging",
    "we train a tiny language model",
    "the residual stream tracks progress",
    "attention starts with token representations",
    "we test before we scale",
    "paper notes guide implementation choices",
    "weekly builds record what broke"
]

print(f"Number of sentences: {len(corpus)}")
for i, line in enumerate(corpus[:5], start=1):
    print(f"{i}. {line}")

Number of sentences: 10
1. the model learns from text
2. the tokenizer splits text into pieces
3. byte pair encoding merges common pairs
4. small corpora are good for debugging
5. we train a tiny language model


## 3) Tokenize and Encode the Corpus

Computers cannot train directly on strings, so we convert words into integers.

Pipeline in plain language:
1. Lowercase and split text into word tokens.
2. Build a vocabulary of unique words.
3. Assign each word an integer ID.
4. Replace every token with its ID.

Why this matters:
- The vocabulary defines what the model can represent.
- ID mapping is the bridge between human-readable text and numeric tensors.
- If this step is wrong, everything downstream is wrong too.

What to check in output:
- Vocabulary size is reasonable for this tiny corpus.
- Token examples and ID examples line up logically.
- Common words appear frequently in the encoded stream.

In [3]:
def tokenize_text(lines):
    tokens = []
    for line in lines:
        tokens.extend(line.lower().split())
    return tokens

all_tokens = tokenize_text(corpus)
vocab = sorted(set(all_tokens))

word_to_id = {w: i for i, w in enumerate(vocab)}
id_to_word = {i: w for w, i in word_to_id.items()}

encoded_tokens = [word_to_id[w] for w in all_tokens]

print(f"Vocabulary size: {len(vocab)}")
print("First 12 tokens:", all_tokens[:12])
print("First 12 IDs:", encoded_tokens[:12])

Vocabulary size: 48
First 12 tokens: ['the', 'model', 'learns', 'from', 'text', 'the', 'tokenizer', 'splits', 'text', 'into', 'pieces', 'byte']
First 12 IDs: [38, 21, 19, 13, 37, 38, 41, 33, 37, 17, 26, 6]


## 4) Create Training Examples

A language model learns by seeing context and predicting what comes next.

How we build examples:
- Take a sliding window of length SEQ_LEN from the encoded stream.
- Use that window as input X.
- Use the next token after the window as target y.

Example idea:
- If input is [the, model, learns, from], target might be text.

Why this matters:
- This converts one long token stream into supervised training pairs.
- The model objective becomes concrete: next-token prediction.
- Shapes printed here are important; they confirm data is ready for matrix math.

Debug checklist:
- X should be 2D with shape (num_examples, SEQ_LEN).
- y should be 1D with shape (num_examples,).
- num_examples should be len(tokens) - SEQ_LEN.

In [4]:
def build_examples(token_ids, seq_len):
    X, y = [], []
    for i in range(len(token_ids) - seq_len):
        X.append(token_ids[i : i + seq_len])
        y.append(token_ids[i + seq_len])
    return np.array(X, dtype=np.int64), np.array(y, dtype=np.int64)

X, y = build_examples(encoded_tokens, SEQ_LEN)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Sample input IDs:", X[0])
print("Sample target ID:", y[0], "->", id_to_word[y[0]])

X shape: (50, 4)
y shape: (50,)
Sample input IDs: [38 21 19 13]
Sample target ID: 37 -> text


## 5) Define the Model

We define a tiny neural baseline using NumPy only.

Model structure:
- Embedding lookup: turn token IDs into dense vectors.
- Context pooling: average vectors across the SEQ_LEN positions.
- Output layer: project pooled vector to vocabulary logits.
- Softmax: convert logits to probabilities.
- Cross-entropy: measure how wrong predictions are.

Why this is a good starter model:
- It is simple enough to understand end-to-end.
- It still captures the core training ingredients used in larger models.
- You can inspect every tensor shape and gradient path manually.

Important intuition:
- Embeddings learn useful coordinates for words.
- The output projection maps context features back into token space.
- Cross-entropy gets smaller when the model assigns higher probability to the true next token.

In [13]:
vocab_size = len(vocab)

params = {
    "E": 0.01 * np.random.randn(vocab_size, EMBED_DIM),   # Embeddings
    "W": 0.01 * np.random.randn(EMBED_DIM, vocab_size),  # Output projection
    "b": np.zeros(vocab_size),                            # Output bias
}


def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exps = np.exp(shifted)
    return exps / np.sum(exps, axis=1, keepdims=True)


def forward(batch_X, model_params):
    # batch_X shape: (batch, seq_len)
    embedded = model_params["E"][batch_X]                # (batch, seq_len, embed_dim)
    hidden = embedded.mean(axis=1)                       # (batch, embed_dim)
    logits = hidden @ model_params["W"] + model_params["b"]
    probs = softmax(logits)
    cache = (embedded, hidden, probs)
    return logits, probs, cache


def cross_entropy_loss(probs, batch_y):
    n = len(batch_y)
    picked = probs[np.arange(n), batch_y]
    return -np.mean(np.log(picked + 1e-12))

## 6) Train the Model

This is the learning loop where parameters are updated.

What happens each batch:
1. Forward pass: compute probabilities for next token.
2. Loss: compute cross-entropy error.
3. Backward pass: compute gradients for E, W, and b.
4. Update: subtract learning_rate * gradient.

What to expect in logs:
- Loss should generally trend downward over epochs.
- Tiny fluctuations are normal because of shuffled mini-batches.

How to debug if loss does not drop:
- Learning rate may be too high or too low.
- Data shapes may be wrong.
- Gradient update signs may be flipped.
- Input-target pairs may be built incorrectly.

Why this section matters:
- This is your first full loop from token IDs to learned parameters.
- Understanding this makes larger training pipelines much easier later.

In [14]:
def one_hot(indices, depth):
    out = np.zeros((len(indices), depth))
    out[np.arange(len(indices)), indices] = 1.0
    return out


def train_epoch(full_X, full_y, model_params, batch_size, lr):
    n = len(full_X)
    order = np.random.permutation(n)
    X_shuf = full_X[order]
    y_shuf = full_y[order]

    epoch_loss = 0.0
    steps = 0

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_X = X_shuf[start:end]
        batch_y = y_shuf[start:end]
        bs = len(batch_X)

        logits, probs, cache = forward(batch_X, model_params)
        embedded, hidden, _ = cache

        loss = cross_entropy_loss(probs, batch_y)
        epoch_loss += loss
        steps += 1

        # Backprop through softmax + cross-entropy
        dlogits = probs.copy()
        dlogits[np.arange(bs), batch_y] -= 1.0
        dlogits /= bs

        dW = hidden.T @ dlogits
        db = dlogits.sum(axis=0)
        dhidden = dlogits @ model_params["W"].T

        # hidden = mean(embedded across seq positions)
        dembedded = np.repeat((dhidden / SEQ_LEN)[:, None, :], SEQ_LEN, axis=1)

        dE = np.zeros_like(model_params["E"])
        for i in range(bs):
            for t in range(SEQ_LEN):
                token_id = batch_X[i, t]
                dE[token_id] += dembedded[i, t]

        # Parameter updates
        model_params["W"] -= lr * dW
        model_params["b"] -= lr * db
        model_params["E"] -= lr * dE

    return epoch_loss / max(steps, 1)


loss_history = []
for epoch in range(1, EPOCHS + 1):
    avg_loss = train_epoch(X, y, params, BATCH_SIZE, LEARNING_RATE)
    loss_history.append(avg_loss)

    if epoch == 1 or epoch % 25 == 0:
        print(f"Epoch {epoch:3d} | loss = {avg_loss:.4f}")

print("Training complete.")

Epoch   1 | loss = 3.8759
Epoch  25 | loss = 3.8205
Epoch  50 | loss = 3.8331
Epoch  75 | loss = 3.7729
Epoch 100 | loss = 3.8173
Epoch 125 | loss = 3.8223
Epoch 150 | loss = 3.8143
Epoch 175 | loss = 3.7973
Epoch 200 | loss = 3.7761
Epoch 225 | loss = 3.7285
Epoch 250 | loss = 3.5716
Training complete.


## 7) Evaluate on Sample Inputs

After training, we test whether the model gives plausible next-word predictions.

Evaluation strategy:
- Provide a short prompt.
- Convert prompt to IDs.
- Run forward pass.
- Show top-k most likely next tokens.

What a good result looks like here:
- Not perfect grammar, because the dataset is tiny.
- But predictions should reflect repeated patterns in your corpus.

Also included:
- Basic sanity checks, including that loss decreased and prediction output shape is correct.

Why this matters:
- Training metrics alone can be misleading.
- Looking at actual predicted tokens gives fast qualitative feedback.

In [15]:
def predict_next(prompt_words, model_params, top_k=5):
    prompt_words = [w.lower() for w in prompt_words]
    # Left-pad/truncate to SEQ_LEN
    if len(prompt_words) < SEQ_LEN:
        prompt_words = ([vocab[0]] * (SEQ_LEN - len(prompt_words))) + prompt_words
    else:
        prompt_words = prompt_words[-SEQ_LEN:]

    ids = np.array([[word_to_id.get(w, 0) for w in prompt_words]], dtype=np.int64)
    _, probs, _ = forward(ids, model_params)
    p = probs[0]

    top_ids = np.argsort(p)[-top_k:][::-1]
    return [(id_to_word[i], float(p[i])) for i in top_ids]


prompts = [
    ["the", "tokenizer", "splits", "text"],
    ["we", "test", "before", "we"],
    ["paper", "notes", "guide", "implementation"],
]

for prompt in prompts:
    preds = predict_next(prompt, params, top_k=5)
    print(f"Prompt: {' '.join(prompt)}")
    print("Top predictions:", preds)
    print("-")

Prompt: the tokenizer splits text
Top predictions: [('text', 0.06149976965919055), ('the', 0.05076628742370995), ('we', 0.047062944312817524), ('stream', 0.028938793465982077), ('splits', 0.026923499870275532)]
-
Prompt: we test before we
Top predictions: [('we', 0.08777759307045363), ('the', 0.030110054805103932), ('scale', 0.028102965128555098), ('tiny', 0.027856918945274547), ('text', 0.027294631989237716)]
-
Prompt: paper notes guide implementation
Top predictions: [('we', 0.05890665486601894), ('the', 0.03352151141307998), ('text', 0.03153447222238915), ('implementation', 0.02546746930781127), ('weekly', 0.02346105783043188)]
-


In [8]:
# Basic sanity checks
assert len(loss_history) == EPOCHS
assert loss_history[-1] < loss_history[0], "Expected loss to decrease over training"

test_prompt = ["the", "model", "learns", "from"]
top = predict_next(test_prompt, params, top_k=3)
assert len(top) == 3

print("Sanity checks passed.")

Sanity checks passed.


## 8) Save and Reload Artifacts

A workflow is not complete until you can persist and restore outputs.

What we save:
- Vocabulary mappings (word_to_id and id_to_word) as JSON.
- Trained model parameters as a pickle file.

What we verify after saving:
- Files exist and load successfully.
- Reloaded vocabulary size matches original.
- Reloaded parameter keys are intact.

Why this matters:
- Reproducibility: you can continue work without retraining every time.
- Portability: saved vocab/model can be used in scripts outside this notebook.
- Good engineering habit: training and deployment paths should connect cleanly.

In [16]:
vocab_path = ARTIFACT_DIR / "toy_vocab.json"
params_path = ARTIFACT_DIR / "toy_model.pkl"

with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump({"word_to_id": word_to_id, "id_to_word": id_to_word}, f, indent=2)

with open(params_path, "wb") as f:
    pickle.dump(params, f)

print(f"Saved vocab to: {vocab_path}")
print(f"Saved params to: {params_path}")

with open(vocab_path, "r", encoding="utf-8") as f:
    loaded_vocab = json.load(f)

with open(params_path, "rb") as f:
    loaded_params = pickle.load(f)

print("Reload successful.")
print("Loaded vocab size:", len(loaded_vocab["word_to_id"]))
print("Loaded parameter keys:", list(loaded_params.keys()))

Saved vocab to: artifacts\toy_vocab.json
Saved params to: artifacts\toy_model.pkl
Reload successful.
Loaded vocab size: 48
Loaded parameter keys: ['E', 'W', 'b']
